# SBI feature selection — choosing the summary-stat vector

**Goal.** Pick a scalar summary-stat vector for SBI that

1. **tracks the update matrix** — the UM is order-specific and is what currently discriminates BE vs SC, so a scalar that correlates with UM cells carries that discriminative signal in an *order-invariant* form SBI can use; and
2. **recovers the model parameters**,

while dropping redundant stats (keep one of `x` / `x_rate`).

**Substrate.** One cohort of BE and SC simulations (all parameters varied), uniform stimuli, `N_TRIALS` as a single mega-session per simulated animal.

**Workflow.** simulate once (cached) → three diagnostics (UM-tracking · param-recovery · redundancy) → curate a shortlist → validate with `select_stats` → write `SBI_STATS`.

*Note:* the simulate cell is slow (a full `N_SIMS=2000 × N_TRIALS=2000` cohort with the history stats is tens of minutes locally); it caches to a `.pkl`, so everything downstream reloads from disk. This is torch-free.

In [ ]:
import sys, time, pickle
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# repo root on the path (works wherever the notebook sits)
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'inference').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from sound_categorisation.validation.feature_diagnostics import (
    simulate_selection_cohort, um_scalar_correlation, stat_correlation,
    stat_individual_power, select_stats)
from sound_categorisation.inference.constants import SBI_STATS

# ── config ───────────────────────────────────────────────────────────────────
DISTRIBUTION = 'uniform'
N_SIMS   = 2000          # BE and SC each — rows for the correlations / recovery
N_TRIALS = 2000          # one mega-session per simulated animal
SCORER   = 'gbm'         # 'gbm' (default) | 'rf' | 'ridge' (fast iteration)
SEED     = 0
POOL = [
    'accuracy', 'psychometric', 'stimulus_sensitivity', 'hard_accuracy',
    'easy_accuracy', 'side_bias', 'choice_entropy', 'pse',
    'recency', 'stimulus_recency', 'recency_divergence',
    'win_stay', 'win_stay_rate', 'lose_shift', 'perseveration',
    'choice_autocorr', 'logistic_history',
]
CACHE = ROOT / 'notebooks' / f'_cohort_{DISTRIBUTION}_{N_SIMS}x{N_TRIALS}.pkl'
print(f"current SBI_STATS ({len(SBI_STATS)}): {SBI_STATS}")
print(f"candidate pool  ({len(POOL)}): {POOL}")

## 0 · Simulate the cohort (slow — cached)

Runs the BE/SC simulator once and caches to disk. **Delete the `.pkl` to force a re-run** (e.g. after changing `POOL`, `N_SIMS` or `N_TRIALS`).

In [ ]:
if CACHE.exists():
    cohort = pickle.loads(CACHE.read_bytes())
    print(f"loaded cached cohort: {CACHE.name}")
else:
    t = time.time()
    cohort = simulate_selection_cohort(distribution=DISTRIBUTION, n_sims=N_SIMS,
                                       n_trials=N_TRIALS, stat_pool=POOL, seed=SEED)
    CACHE.write_bytes(pickle.dumps(cohort))
    print(f"simulated + cached in {time.time()-t:.0f}s -> {CACHE.name}")
print(f"n_valid = {cohort.n_valid}")
print(f"params  BE = {cohort.param_names['be']}")
print(f"params  SC = {cohort.param_names['sc']}")
print(f"UM cells = {len(cohort.um_cols)} | predictor stats = {len(cohort.pred_cols())}")

## 1 · UM tracking — do scalars carry the UM's discriminative content?

For each UM cell, the **max |r| over scalars** says whether *any* scalar tracks it. A UM cell where every |r| is low is discriminative structure the scalar vector cannot see (e.g. the SC stripe) — a warning that identity will suffer no matter which scalars we pick.

In [ ]:
DARK = 0.3   # a UM cell whose best scalar |r| < DARK is poorly tracked

um = {m: um_scalar_correlation(cohort, m)['corr'] for m in ('be', 'sc')}
for m in ('be', 'sc'):
    C = um[m]
    best = pd.DataFrame({'max_|r|': C.max(1), 'best_stat': C.idxmax(1)})
    n_dark = int((C.max(1) < DARK).sum())
    print(f"[{m.upper()}] {C.shape[0]} UM cells, {n_dark} poorly tracked (best |r| < {DARK})")
    display(best.sort_values('max_|r|').head(12).style.background_gradient(
        subset=['max_|r|'], cmap='RdYlGn', vmin=0, vmax=1).set_caption(
        f'{m.upper()}: worst-tracked UM cells'))

# strongest UM-tracking scalars overall (max |r| over UM cells)
tracker = pd.DataFrame({'BE': um['be'].max(0), 'SC': um['sc'].max(0)}).sort_values('SC', ascending=False)
display(tracker.style.background_gradient(cmap='Greens', vmin=0, vmax=1).set_caption(
    'strongest UM-tracking scalar (max |r| over UM cells)'))

## 2 · Parameter recovery + BE/SC identity

`stat_individual_power` scores each stat on its own: a direct BE-vs-SC identity AUC, and per-parameter recovery R² (per model). A stat can be individually strong yet jointly redundant — that's what §3 and §4 are for.

In [ ]:
ip = stat_individual_power(cohort, scorer=SCORER, seed=SEED)

auc = ip['identity'].sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
auc.plot.barh(ax=ax); ax.axvline(0.5, color='0.6', ls='--')
ax.set_xlabel('BE-vs-SC identity AUC'); ax.invert_yaxis(); plt.tight_layout(); plt.show()

# per-param recovery R² (max over the two models: a param is recovered if EITHER model's fit recovers it)
rec = pd.concat({m: ip['recovery'][m] for m in ('be', 'sc')}).groupby(level=1).max().reindex(POOL)
display(rec.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=0.8, axis=None).format('{:.2f}').set_caption(
    'recovery R² (max over models): stats (rows) x parameters (cols)'))

best_rec = pd.DataFrame({'best_stat': rec.idxmax(0), 'R2': rec.max(0)})
display(best_rec.style.background_gradient(subset=['R2'], cmap='RdYlGn', vmin=0, vmax=0.8).set_caption(
    'best single-stat recoverer per parameter'))

## 3 · Redundancy — drop near-duplicate stats

Pairs with |r| near 1 are interchangeable; keep the stronger recoverer / UM-tracker of each pair.

In [ ]:
REDUN = 0.9
Cred = stat_correlation(cohort, 'sc')['corr']      # stat x stat (SC; BE is similar)
display(Cred.style.background_gradient(cmap='Reds', vmin=0, vmax=1, axis=None).format('{:.2f}').set_caption(
    'stat-stat |r| (SC) — look for blocks near 1'))

pairs = [(a, b, float(Cred.loc[a, b])) for i, a in enumerate(Cred.index) for b in Cred.index[i+1:]
         if Cred.loc[a, b] > REDUN]
print(f"redundant pairs (|r| > {REDUN}) — keep one of each:")
for a, b, r in sorted(pairs, key=lambda t: -t[2]):
    print(f"  {a:24} ~ {b:24} |r| = {r:.2f}")

## 4 · Curate the shortlist

Start from the auto-suggestion — each parameter's best recoverer ∪ the strongest UM-trackers, minus redundant duplicates — then **edit `SHORTLIST` by hand** using the tables above. Keep it to ~10-12 stats so `select_stats` can run exhaustively.

In [ ]:
K_UM = 6
auto = set(best_rec['best_stat']) | set(tracker.head(K_UM).index)
for a, b, r in pairs:                               # drop the weaker of each redundant pair
    if a in auto and b in auto:
        drop = a if rec.max(1).get(a, 0) < rec.max(1).get(b, 0) else b
        auto.discard(drop)
auto = [s for s in POOL if s in auto]
print("auto-suggested shortlist:", auto)

# --- EDIT BY HAND ---
SHORTLIST = list(auto)
print("SHORTLIST:", SHORTLIST)

## 5 · Validate the shortlist

`select_stats` searches subsets of the shortlist for joint identity AUC + parameter recovery. Its headline `r2_min` is the **min over *all* parameters**, so it looks bad whenever a parameter is inherently unrecoverable from scalars — read the identity AUC and the *per-parameter* recovery of the selected subset (last table) separately, rather than trusting the single number.

In [ ]:
from sound_categorisation.validation.feature_diagnostics import _cv_r2

sel = select_stats(cohort, shortlist=SHORTLIST, scorer=SCORER,
                   method='auto', identity_floor=0.9, seed=SEED)
print(f"search method: {sel['method']}")
display(sel['best_by_k'][['k', 'identity_auc', 'r2_min']].round(3))
print("SELECTED subset:", sorted(sel['selected']))
print(f"  identity AUC = {sel['selected_identity_auc']:.3f}   min param R² = {sel['selected_r2_min']:.3f}")

# per-parameter recovery of the SELECTED subset (not just the min)
cols = sorted({c for g in sel['selected'] for c in cohort.pred_groups[g]})
rows = {m: {p: _cv_r2(SCORER, cohort.X[m][:, cols], cohort.theta[m][:, j], 5, SEED)
            for j, p in enumerate(cohort.param_names[m])} for m in ('be', 'sc')}
display(pd.DataFrame(rows).style.background_gradient(cmap='RdYlGn', vmin=0, vmax=0.8, axis=None).format('{:.2f}').set_caption(
    'selected subset — per-parameter recovery R² (per model)'))

## 6 · Chosen vector

In [ ]:
SBI_STATS_NEW = sorted(sel['selected'])
print("Chosen SBI_STATS:")
print(SBI_STATS_NEW)
print(f"\n({len(SBI_STATS_NEW)} stats) — paste into inference/constants.py as SBI_STATS, or keep iterating SHORTLIST above.")